# Phase 6 v1 — Legal abstractive summarization / five-method ablation

**Source architecture:** `phase-2-v1 (8).ipynb`, code cell starting `%%writefile llama_scratch.py`. Preserved original Llama-style decoder LM: GQA, RoPE, RMSNorm, SwiGLU, tied LM head. Adaptation modules are new, handwritten PyTorch implementations. **No pretrained third-party LM weights.**

**Important:** An authentic Phase-1 checkpoint + exact Phase-1 tokenizer files must be supplied. Smoke mode also requires this checkpoint, and is NOT a random-weight demo. This notebook provides five PEFT method comparisons on one Phase-1 model; the full competition also mandates four different Phase-1 architectures and a distillation experiment. Those are **not** supplied by this five-method ablation.

Execute cells in order. Full training is GPU-intensive. Best-checkpoint selection uses a train-derived validation split; final inference uses best validation checkpoint for each method (not refit on validation). `full` never limits record count.

## 1. Editable settings — FIRST LINE: `MODE = "smoke"`

In [ ]:
MODE = "full"  # "smoke" | "full"
# Update paths to your Kaggle/Colab environment. Input dataset is not modified.
HF_DATASET_REPO   = "Exploration-Lab/CS-781-Capstone"
TRAIN_REMOTE      = "Phase_6_Summ/Train/train.json"
TEST_REMOTE       = "Phase_6_Summ/Test/test.json"
TOKENIZER_REPO    = "DumbLad/legal-bpe-tokenizer-phase1"
PHASE1_CHECKPOINT = "/kaggle/input/datasets/koooshiieee/phase2v1/final_model.pt"
TOKENIZER_DIR     = "/kaggle/working/tok_phase1"
OUT_DIR = "/kaggle/working/phase6_v1"

SEED = 67
METHODS = ["houlsby", "pfeiffer", "lora", "adalora", "lorma"]
# Dataset-specific validation and context, max context must fit Phase-1 RoPE cache.
VAL_FRAC = 0.05
CHUNK_OVERLAP = 64
MAX_INPUT_TOKENS = 768
MAX_TARGET_TOKENS = 384
SUMMARY_NEW_TOKENS = 384
RANK = 8
ADAPTER_BOTTLENECK = 32
LR = 2e-4
WEIGHT_DECAY = 0.01
EPOCHS = 1 if MODE == "smoke" else 2
BATCH_SIZE = 1
GRAD_ACCUM = 1 if MODE == "smoke" else 8
SMOKE_TRAIN_DOCS = 4
SMOKE_VAL_DOCS = 2
SMOKE_TEST_DOCS = 2  # only smoke; full always exports every test doc
USE_AMP = True
assert MODE in ("smoke", "full")

LOG_EVERY = 25 if MODE=='full' else 1       # optimizer steps between progress lines
EVAL_EVERY = 150 if MODE=='full' else 0     # optimizer steps between mid-epoch val
EVAL_LIMIT = 48                             # val chunks used for mid-epoch val
MAX_EX_PER_EPOCH = 8000                     # set from benchmark C
TIME_LIMIT_MIN = 25                       # per-method wall-clock cap, from benchmark C
TRAIN_SPECIAL_ROWS = [259]                  # </s>; add 258 (<s>) if you want
VAL_GEN_DOCS = 100       # docs used for ROUGE per method
VAL_LOSS_LIMIT = 600     # val chunks used for best-checkpoint selection
GEN_BS = 32              # drop to 16 on CUDA OOM
K_CHUNKS = 3
DRAFT_TOK = 96
FINAL_TOK = 192

# assert all(isinstance(p['id'],str) and p['summary'].strip() for p in predictions), 'evaluator rejects empty summaries / non-str ids'

## 2. Install, imports and reproducibility — FIRST LINE: `# Standard dependencies`

In [ ]:
# Standard dependencies (no PEFT, adapters, transformers model or fine-tuning helper)
%pip -q install regex tqdm rouge-score # bert-score
import os, json, math, time, copy, random, re, gc, pathlib, itertools
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import asdict
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('device', DEVICE, 'mode',MODE)

In [ ]:
# HF auth, data + tokenizer download, output repo
from huggingface_hub import HfApi, hf_hub_download, login
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF"); os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

TRAIN_JSON = hf_hub_download(HF_DATASET_REPO, TRAIN_REMOTE, repo_type="dataset", token=HF_TOKEN)
TEST_JSON  = hf_hub_download(HF_DATASET_REPO, TEST_REMOTE,  repo_type="dataset", token=HF_TOKEN)
for f in ["vocab.json", "merges.json", "tokenizer_config.json"]:
    hf_hub_download(TOKENIZER_REPO, f, repo_type="model", local_dir=TOKENIZER_DIR, token=HF_TOKEN)

hf_api = HfApi(token=HF_TOKEN)
HF_OUT_REPO = f"{hf_api.whoami()['name']}/legal-summ-phase6-v1" + ("-smoke" if MODE == "smoke" else "")
hf_api.create_repo(HF_OUT_REPO, repo_type="model", private=True, exist_ok=True)

def hf_push(local_path, path_in_repo, tries=3):
    for k in range(tries):
        try:
            hf_api.upload_file(path_or_fileobj=local_path, path_in_repo=path_in_repo,
                               repo_id=HF_OUT_REPO, repo_type="model")
            print("uploaded ->", HF_OUT_REPO, path_in_repo); return True
        except Exception as e:
            print(f"upload attempt {k+1} failed: {e}"); time.sleep(15)
    print("UPLOAD FAILED:", path_in_repo); return False

## 3. Exact original Phase-2 underlying Phase-1 Llama LM (unchanged) — FIRST LINE: `"""`

In [ ]:
"""
Llama-style decoder-only LM, written from scratch (no HF / no llama library).

Pieces: RMSNorm, RoPE (or ALiBi), grouped-query attention (GQA; MQA = n_kv_heads 1,
MHA = n_kv_heads == n_heads), SwiGLU MLP, pre-norm residual blocks, tied embeddings,
KV-cache generation with left-padded batches.

Only torch tensors/autograd/nn.Linear/nn.Embedding are used as building blocks.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------- config
@dataclass
class Config:
    vocab_size: int = 33000
    d_model: int = 1024
    n_layers: int = 16
    n_heads: int = 16
    n_kv_heads: int = 4          # 16 query heads share 4 KV heads (4 queries per KV head)
    ffn_hidden: int = 2560
    max_seq_len: int = 1024      # training context
    rope_cache_len: int = 4096   # RoPE table length (>= prompt + generated tokens)
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    tie_embeddings: bool = True
    pos_enc: str = "rope"        # "rope" | "alibi"
    attn_impl: str = "manual"    # "manual" (hand-written softmax(QK^T)V) | "sdpa" (torch fused kernel, same math)


PRESETS = {
    # exact totals with vocab 33000, tied embeddings
    "target_150m": dict(d_model=896, n_layers=14, n_heads=14, n_kv_heads=2, ffn_hidden=2304), # ~149.8M 
    "base":  dict(d_model=1024, n_layers=16, n_heads=16, n_kv_heads=4, ffn_hidden=2560),  # ~201.6M
    "small": dict(d_model=1024, n_layers=12, n_heads=16, n_kv_heads=4, ffn_hidden=2816),  # ~169.1M
    "tiny":  dict(d_model=64,   n_layers=2,  n_heads=4,  n_kv_heads=2, ffn_hidden=128),   # smoke tests
}


def analytic_param_count(c: Config) -> int:
    """Closed-form parameter count (use it to size a model before building it)."""
    hd = c.d_model // c.n_heads
    attn = 2 * c.d_model * c.n_heads * hd + 2 * c.d_model * c.n_kv_heads * hd   # Wq,Wo + Wk,Wv
    mlp = 3 * c.d_model * c.ffn_hidden                                           # gate, up, down
    per_layer = attn + mlp + 2 * c.d_model                                       # + 2 RMSNorm
    total = per_layer * c.n_layers + c.vocab_size * c.d_model + c.d_model        # + embedding + final norm
    if not c.tie_embeddings:
        total += c.vocab_size * c.d_model
    return total


def count_params(model, verbose=True) -> int:
    """model.parameters() yields a tied (shared) tensor only once, so tied weights are counted once."""
    total = sum(p.numel() for p in model.parameters())
    if verbose:
        emb = model.tok_emb.weight.numel()
        blocks = sum(p.numel() for p in model.layers.parameters())
        print(f"total {total/1e6:.1f}M | transformer blocks {blocks/1e6:.1f}M | embedding {emb/1e6:.1f}M "
              f"({'tied with output head' if model.cfg.tie_embeddings else 'output head separate'}) "
              f"| analytic {analytic_param_count(model.cfg)/1e6:.1f}M")
    return total


# ----------------------------------------------------------------------------- RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()                                   # always normalise in fp32
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)


# ----------------------------------------------------------------------------- RoPE
def build_rope_cache(head_dim, max_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))   # (hd/2,)
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)                                                # (max_len, hd/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x: (B, H, T, hd); cos/sin: (T, hd/2). Rotates the two halves of each head vector (Llama layout)."""
    half = x.shape[-1] // 2
    xf = x.float()
    x1, x2 = xf[..., :half], xf[..., half:]
    cos = cos[None, None]
    sin = sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


# ----------------------------------------------------------------------------- ALiBi
def get_alibi_slopes(n_heads):
    def pow2(n):
        start = 2 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_heads).is_integer():
        s = pow2(n_heads)
    else:
        c = 2 ** math.floor(math.log2(n_heads))
        s = pow2(c) + pow2(2 * c)[0::2][: n_heads - c]
    return torch.tensor(s, dtype=torch.float32)


# ----------------------------------------------------------------------------- GQA
def repeat_kv(x, n_rep):
    """(B, n_kv, S, hd) -> (B, n_kv*n_rep, S, hd); query head h uses kv head h // n_rep."""
    if n_rep == 1:
        return x
    B, n_kv, S, hd = x.shape
    return x[:, :, None].expand(B, n_kv, n_rep, S, hd).reshape(B, n_kv * n_rep, S, hd)


class GQAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.nh, self.nkv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.n_rep = self.nh // self.nkv
        self.impl = cfg.attn_impl
        self.wq = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)    # (B, nh,  T, hd)
        k = self.wk(x).view(B, T, self.nkv, self.hd).transpose(1, 2)   # (B, nkv, T, hd)
        v = self.wv(x).view(B, T, self.nkv, self.hd).transpose(1, 2)

        if cos is not None:                                            # RoPE on q and k (not v)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past is not None:                                           # cache holds only nkv heads
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        new_cache = (k, v) if use_cache else None
        S = k.shape[2]

        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # relative distance of every (query, key) pair; keys j <= query position are visible
        qpos = torch.arange(S - T, S, device=x.device)
        kpos = torch.arange(S, device=x.device)
        rel = qpos[:, None] - kpos[None, :]                            # (T, S)
        allowed = (rel >= 0)[None, None]                               # (1, 1, T, S)
        if key_valid is not None:                                      # hide left-padding keys
            allowed = allowed & key_valid[:, None, None, :]
        bias = None
        if alibi_slopes is not None:
            bias = -alibi_slopes.view(1, -1, 1, 1) * rel.clamp(min=0).float()[None, None]   # (1, nh, T, S)

        if self.impl == "sdpa" and key_valid is None and T == S:      # training fast path
            if bias is None:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            else:
                m = bias.masked_fill(~allowed, torch.finfo(q.dtype).min).to(q.dtype)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        else:                                                          # hand-written attention
            scores = ((q * (self.hd ** -0.5)) @ k.transpose(-2, -1)).float()   # (B, nh, T, S)
            if bias is not None:
                scores = scores + bias
            scores = scores.masked_fill(~allowed, -1e30)               # finite, so a fully-masked pad row can't give NaN
            probs = torch.softmax(scores, dim=-1).to(v.dtype)
            out = probs @ v                                            # (B, nh, T, hd)

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.wo(out), new_cache


# ----------------------------------------------------------------------------- SwiGLU
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden, bias=False)
        self.w_up = nn.Linear(d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


# ----------------------------------------------------------------------------- block + model
class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg.d_model, cfg.ffn_hidden)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        h, new_cache = self.attn(self.attn_norm(x), cos, sin, alibi_slopes, key_valid, past, use_cache)
        x = x + h
        x = x + self.mlp(self.mlp_norm(x))
        return x, new_cache


class LlamaLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.pos_enc in ("rope", "alibi") and cfg.attn_impl in ("manual", "sdpa")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        if cfg.pos_enc == "rope":
            cos, sin = build_rope_cache(cfg.d_model // cfg.n_heads, cfg.rope_cache_len, cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
            self.alibi_slopes = None
        else:
            self.rope_cos = self.rope_sin = None
            self.register_buffer("alibi_slopes", get_alibi_slopes(cfg.n_heads), persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():       # GPT-2 style: shrink the residual-branch output projections
            if name.endswith("wo.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False, cache=None, start_pos=0,
                key_valid=None, all_logits=False):
        """
        training:  model(x, targets=y) -> scalar loss
        inference: model(idx, use_cache=True, cache=..., start_pos=..., key_valid=...) -> (logits, cache)
                   logits are for the last position only unless all_logits=True.
        """
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos = sin = None
        if self.cfg.pos_enc == "rope":
            assert start_pos + T <= self.cfg.rope_cache_len, "sequence longer than the RoPE table"
            cos = self.rope_cos[start_pos:start_pos + T]
            sin = self.rope_sin[start_pos:start_pos + T]

        new_caches = []
        for i, layer in enumerate(self.layers):
            past = cache[i] if cache is not None else None
            x, c = layer(x, cos, sin, self.alibi_slopes, key_valid, past, use_cache)
            new_caches.append(c)
        x = self.norm(x)

        if targets is not None:
            logits = self.lm_head(x)
            return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        logits = self.lm_head(x if all_logits else x[:, -1:, :])
        return logits, (new_caches if use_cache else None)


# ----------------------------------------------------------------------------- generation
@torch.no_grad()
def generate(model, prompts, max_new_tokens, eos_id, pad_id, temperature=0.1, top_k=30,
             top_p=0.95, rep_penalty=1.2, ban_eos=True):
    """
    prompts: list of lists of token ids (any lengths). Left-pads them into one batch, runs one prefill,
    then decodes with a KV cache. Returns a list of lists of generated ids (EOS / padding removed).
    temperature <= 0 -> greedy.
    """
    model.eval()
    device = next(model.parameters()).device
    V = model.cfg.vocab_size
    B = len(prompts)
    L = max(len(p) for p in prompts)
    ids = torch.full((B, L), pad_id, dtype=torch.long, device=device)
    key_valid = torch.zeros(B, L, dtype=torch.bool, device=device)
    seen = torch.zeros(B, V, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        t = torch.tensor(p, dtype=torch.long, device=device)
        ids[b, L - len(p):] = t
        key_valid[b, L - len(p):] = True
        seen[b, t] = True

    out = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    amp = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda"))
    with amp:
        logits, cache = model(ids, use_cache=True, key_valid=key_valid, start_pos=0)
        for step in range(max_new_tokens):
            lg = logits[:, -1, :].float()
            if ban_eos:
                lg[:, eos_id] = float("-inf")
            if rep_penalty != 1.0:                                     # CTRL-style repetition penalty
                pen = torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty)
                lg = torch.where(seen, pen, lg)
            if temperature <= 0:
                nxt = lg.argmax(-1)
            else:
                lg = lg / temperature
                if top_k and top_k > 0:
                    kth = torch.topk(lg, min(top_k, V)).values[:, -1:]
                    lg = lg.masked_fill(lg < kth, float("-inf"))
                if top_p < 1.0:
                    sl, si = torch.sort(lg, descending=True, dim=-1)
                    pr = torch.softmax(sl, dim=-1)
                    drop = (pr.cumsum(-1) - pr) > top_p                # keep the token that crosses top_p
                    sl = sl.masked_fill(drop, float("-inf"))
                    lg = torch.full_like(lg, float("-inf")).scatter(1, si, sl)
                nxt = torch.multinomial(torch.softmax(lg, dim=-1), 1).squeeze(1)

            nxt = torch.where(finished, torch.full_like(nxt, pad_id), nxt)
            for b in range(B):
                if not finished[b] and nxt[b].item() != eos_id:
                    out[b].append(int(nxt[b].item()))
            finished = finished | (nxt == eos_id)
            seen.scatter_(1, nxt[:, None], True)
            if bool(finished.all()):
                break
            key_valid = torch.cat([key_valid, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)
            logits, cache = model(nxt[:, None], use_cache=True, cache=cache,
                                  start_pos=L + step, key_valid=key_valid)
    return out

## 4. Exact Phase-1 tokenizer implementation (unchanged) — FIRST LINE: `#!/usr/bin/env python3`

In [ ]:
#!/usr/bin/env python3
import os, json, regex
from typing import Dict, List, Tuple
from collections import Counter, defaultdict
from tqdm.auto import tqdm

import regex


_PAT = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")


class PureByteLevelBPETokenizer:
    def __init__(self, vocab_size: int = 33000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]

        self.encoder: Dict[bytes, int] = {}
        self.decoder: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], bytes] = {}       # pair -> merged token
        self.merge_ranks: Dict[Tuple[bytes, bytes], int] = {}    # pair -> order learned (lower = earlier)
        self._cache: Dict[str, List[int]] = {}

        self._init_base_vocab()

    def _init_base_vocab(self):
        for b in range(256):
            b_bytes = bytes([b])
            self.encoder[b_bytes] = b
            self.decoder[b] = b_bytes

        # special tokens
        for idx, token in enumerate(self.special_tokens, start=256):
            b_bytes = token.encode("utf-8")
            self.encoder[b_bytes] = idx
            self.decoder[idx] = b_bytes

    @property
    def pad_token_id(self) -> int:
        return self.encoder["<pad>".encode("utf-8")]

    @property
    def unk_token_id(self) -> int:
        return self.encoder["<unk>".encode("utf-8")]

    @property
    def bos_token_id(self) -> int:
        return self.encoder["<s>".encode("utf-8")]

    @property
    def eos_token_id(self) -> int:
        return self.encoder["</s>".encode("utf-8")]

    # ------------------------------------------------------------------ training
    @staticmethod
    def _merge_word(word, a, b, new):
        out, i, n = [], 0, len(word)
        while i < n:
            if i < n - 1 and word[i] == a and word[i + 1] == b:
                out.append(new)
                i += 2
            else:
                out.append(word[i])
                i += 1
        return out

    def train_from_texts(self, texts: List[str], target_vocab_size: int = None, min_word_freq: int = 2):
        if target_vocab_size is None:
            target_vocab_size = self.vocab_size
        num_merges = target_vocab_size - len(self.encoder)
        print(f"\nTraining byte level BPE from scratch. Target merges: {num_merges:,} | Base Vocab: {len(self.encoder)}")

        # 1) pre-tokenize + count unique words (the BPE then runs on this small table, not the raw text)
        word_counts = Counter()
        for text in tqdm(texts, desc="Counting pre-tokens"):
            if text and text.strip():
                word_counts.update(_PAT.findall(text))

        words, freqs = [], []
        for w, c in word_counts.items():
            if c >= min_word_freq:
                words.append([bytes([b]) for b in w.encode("utf-8", "replace")])
                freqs.append(c)
        del word_counts
        print(f"Unique words used for merge learning: {len(words):,}")

        # 2) initial pair statistics (updated incrementally afterwards)
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for i, w in enumerate(tqdm(words, desc="Initial pair counts")):
            for p in zip(w, w[1:]):
                pair_counts[p] += freqs[i]
                where[p].add(i)

        # 3) merge loop
        pbar = tqdm(total=num_merges, desc="Learning BPE Merges", dynamic_ncols=True)
        for _ in range(num_merges):
            if not pair_counts:
                print("No more candidate pairs found to merge.")
                break

            best_pair, freq = max(pair_counts.items(), key=lambda kv: kv[1])
            if freq < 2:
                print(f"Stopping early: Most frequent pair frequency is {freq}")
                break

            a, b = best_pair
            merged_token = a + b
            self.merges[best_pair] = merged_token
            self.merge_ranks[best_pair] = len(self.merge_ranks)
            if merged_token not in self.encoder:          # two merge paths can give the same bytes
                new_id = len(self.encoder)
                self.encoder[merged_token] = new_id
                self.decoder[new_id] = merged_token

            for i in list(where[best_pair]):
                w, f = words[i], freqs[i]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        del pair_counts[p]
                w = self._merge_word(w, a, b, merged_token)
                words[i] = w
                for p in zip(w, w[1:]):
                    pair_counts[p] += f
                    where[p].add(i)
            del where[best_pair]
            pair_counts.pop(best_pair, None)

            pbar.update(1)
            pbar.set_postfix({"vocab_size": len(self.encoder), "top_pair_freq": freq})

        pbar.close()
        self._cache.clear()
        print(f"Training finished! Final Vocab Size: {len(self.encoder):,}")

    # ------------------------------------------------------------------ encode / decode
    def _encode_word(self, word: str) -> List[int]:
        cached = self._cache.get(word)
        if cached is not None:
            return cached
        tokens = [bytes([b]) for b in word.encode("utf-8", "replace")]
        while len(tokens) >= 2:
            best_pair, best_rank = None, None
            for p in zip(tokens, tokens[1:]):
                r = self.merge_ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_pair, best_rank = p, r
            if best_pair is None:
                break
            tokens = self._merge_word(tokens, best_pair[0], best_pair[1], self.merges[best_pair])
        ids = [self.encoder.get(t, self.unk_token_id) for t in tokens]
        if len(self._cache) < 1_000_000:
            self._cache[word] = ids
        return ids

    def encode(self, text: str) -> List[int]:
        ids: List[int] = []
        for word in _PAT.findall(text):
            ids.extend(self._encode_word(word))
        return ids

    def decode(self, token_ids: List[int]) -> str:
        raw_bytes = b"".join(self.decoder.get(i, b"<unk>") for i in token_ids)
        return raw_bytes.decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ save / load
    def save(self, save_dir: str):
        os.makedirs(save_dir, exist_ok=True)
        vocab_path = os.path.join(save_dir, "vocab.json")
        merges_path = os.path.join(save_dir, "merges.json")
        config_path = os.path.join(save_dir, "tokenizer_config.json")

        # latin1 maps bytes 1:1 onto code points, so any byte string survives JSON
        ser_vocab = {k.decode("latin1"): v for k, v in self.encoder.items()}
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(ser_vocab, f, ensure_ascii=False, indent=2)

        # JSON (not a space-separated text file): tokens can contain spaces, newlines, \r, \xa0 ...
        ser_merges = [[p1.decode("latin1"), p2.decode("latin1")] for (p1, p2) in self.merges]  # in learned order
        with open(merges_path, "w", encoding="utf-8") as f:
            json.dump(ser_merges, f, ensure_ascii=False)

        config = {
            "tokenizer_class": "PureByteLevelBPETokenizer",
            "vocab_size": len(self.encoder),
            "special_tokens": self.special_tokens,
            "pad_token_id": self.pad_token_id,
            "bos_token_id": self.bos_token_id,
            "eos_token_id": self.eos_token_id
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

        print(f"Tokenizer saved successfully to {save_dir}")

    @classmethod
    def load(cls, save_dir: str) -> "PureByteLevelBPETokenizer":
        tok = cls()
        with open(os.path.join(save_dir, "vocab.json"), "r", encoding="utf-8") as f:
            ser_vocab = json.load(f)
        tok.encoder = {k.encode("latin1"): v for k, v in ser_vocab.items()}
        tok.decoder = {v: k.encode("latin1") for k, v in ser_vocab.items()}

        tok.merges, tok.merge_ranks = {}, {}
        with open(os.path.join(save_dir, "merges.json"), "r", encoding="utf-8") as f:
            for rank, (s1, s2) in enumerate(json.load(f)):
                p1, p2 = s1.encode("latin1"), s2.encode("latin1")
                tok.merges[(p1, p2)] = p1 + p2
                tok.merge_ranks[(p1, p2)] = rank
        return tok

## 5. Verify checkpoint, tokenizer and input JSON — FIRST LINE: `def load_records`

In [ ]:
import json, collections
with open(TRAIN_JSON, encoding="utf8") as f: o = json.load(f)
if isinstance(o, dict):
    print("top-level keys:", list(o)[:10])
    for k in ("data","records","examples"):
        if k in o and isinstance(o[k], list): o = o[k]; break
print("n records:", len(o))
print("key sets:", collections.Counter(tuple(sorted(x.keys())) for x in o).most_common(5))
print("summary types:", collections.Counter(type(x.get("summary")).__name__ for x in o))
bad = [x for x in o if not (isinstance(x.get("summary"), str) and x["summary"].strip())]
print("bad count:", len(bad))
for x in bad[:3]: print({k: (str(v)[:80]) for k, v in x.items()})

In [ ]:
def load_records(path, training):
    assert os.path.isfile(path), f"Missing dataset: {path}"
    with open(path, encoding="utf8") as f: obj = json.load(f)
    if isinstance(obj,dict):
        for field in ("data","records","examples"):
            if field in obj and isinstance(obj[field],list): obj=obj[field];break
    assert isinstance(obj,list), "Expected a JSON list of document records"
    out, seen, dropped = [], set(), []
    for x in obj:
        assert isinstance(x,dict) and all(k in x for k in ('id','document'))
        assert isinstance(x['document'],str) and x['document'].strip()
        assert str(x['id']) not in seen, 'Duplicate document id'; seen.add(str(x['id']))
        if training and not (isinstance(x.get('summary'),str) and x['summary'].strip()):
            dropped.append(str(x['id'])); continue      # skip empty-summary training rows
        out.append(x)
    if dropped: print(f"Dropped {len(dropped)} training records with empty summary: {dropped}")
    return out
train_raw=load_records(TRAIN_JSON,True);test_raw=load_records(TEST_JSON,False)
print('Train / test',len(train_raw),len(test_raw))
for p in ('vocab.json','merges.json','tokenizer_config.json'):
    assert os.path.exists(os.path.join(TOKENIZER_DIR,p)), 'Missing tokenizer: '+p
assert os.path.isfile(PHASE1_CHECKPOINT), 'Phase-1 checkpoint missing. Do not replace with pretrained public LM.'
ck = torch.load(PHASE1_CHECKPOINT,map_location='cpu',weights_only=False)
assert isinstance(ck,dict) and 'model' in ck and 'cfg' in ck, 'Expected checkpoint with model and cfg; adjust this loading cell for your Phase-1 save format.'
cfg=Config(**ck['cfg'])
cfg.attn_impl='sdpa' if DEVICE.type=='cuda' else 'manual'
assert MAX_INPUT_TOKENS+MAX_TARGET_TOKENS+32 <= cfg.rope_cache_len, 'Increase Phase-1 RoPE cache length in cfg or reduce token limits'
assert MAX_INPUT_TOKENS+MAX_TARGET_TOKENS+32 <= cfg.max_seq_len or cfg.rope_cache_len>=MAX_INPUT_TOKENS+MAX_TARGET_TOKENS+32, 'Context incompatible'
tok=PureByteLevelBPETokenizer.load(TOKENIZER_DIR)
print('checkpoint config:',cfg,'tokenizer:',tok.vocab_size)
assert cfg.vocab_size==tok.vocab_size, 'Tokenizer and checkpoint vocab mismatch'
initial_state=ck['model'];del ck
# The exact tokenizer must be checked against your Phase-1 implementation: special-token ids as below.
print('special ids', tok.pad_token_id, tok.unk_token_id, tok.bos_token_id, tok.eos_token_id)
assert [tok.pad_token_id, tok.unk_token_id, tok.bos_token_id, tok.eos_token_id] == [256, 257, 258, 259]

In [ ]:
E=initial_state['tok_emb.weight'].float(); d=E.shape[1]; nr=E.norm(dim=1)
print(f"untrained init row norm ~ {0.02*math.sqrt(d):.3f}")
for nm,i in dict(pad=256,unk=257,bos=258,eos=259,mask=260).items():
    print(f"{nm:5s} id={i} norm={nr[i]:.3f} std={E[i].std():.4f}")
print("ordinary tokens norm median/p5/p95:",nr[300:].median().item(),nr[300:].quantile(.05).item(),nr[300:].quantile(.95).item())
print("cos(eos,mask) =",F.cosine_similarity(E[259],E[260],dim=0).item())
m_=LlamaLM(cfg).to(DEVICE); m_.load_state_dict(initial_state); m_.eval()
pe,pm,rk=[],[],[]
with torch.no_grad(), torch.autocast('cuda',dtype=torch.float16,enabled=DEVICE.type=='cuda'):
    for r in random.Random(0).sample(train_raw,min(30,len(train_raw))):
        ids=tok.encode(r['document'])[-512:]
        lg=m_(torch.tensor([ids],device=DEVICE),all_logits=True)[0][0].float()
        p=lg.log_softmax(-1)[:,259].exp()
        pe.append(p[-1].item()); pm.append(p[:-1].mean().item()); rk.append(int((lg[-1]>lg[-1,259]).sum()))
print("P(EOS | end of doc) median:",np.median(pe)," P(EOS) mid-text mean:",np.mean(pm)," EOS rank at end (median):",int(np.median(rk)))
del m_; gc.collect(); torch.cuda.empty_cache() if DEVICE.type=='cuda' else None

## 6. Tokenization, section-aware long-document chunking and causal summary targets — FIRST LINE: `def enc`

In [ ]:
def enc(s):
    ids=tok.encode(s)
    return ids.ids if hasattr(ids,'ids') else ids
def dec(ids):
    return tok.decode(ids)


PAD, BOS, EOS = tok.pad_token_id, tok.bos_token_id, tok.eos_token_id
assert 0<=EOS<cfg.vocab_size

def segments(text,max_tokens):
    # Section-aware split on paragraph boundaries, then sliding windows for oversize paragraphs.
    paragraphs=re.split(r'\n\s*\n',text.strip())
    if len(paragraphs)==1: paragraphs=re.split(r'(?<=[.!?])\s+(?=[A-Z])',text.strip())
    result=[]; current=[]
    for p in paragraphs:
        ids=enc(p+' ')
        if len(current)+len(ids)>max_tokens and current:
            result.append(current);current=current[-min(CHUNK_OVERLAP,len(current)):]
        while len(ids)>max_tokens:
            take=max_tokens-len(current)
            if take<=0:result.append(current);current=current[-min(CHUNK_OVERLAP,len(current)):];take=max_tokens-len(current)
            current.extend(ids[:take]);ids=ids[take:]
            if ids:result.append(current);current=current[-min(CHUNK_OVERLAP,len(current)):]
        current.extend(ids)
    if current:result.append(current)
    return result or [[]]

PROMPT_PREFIX='Legal judgment:\n'
PROMPT_SUFFIX='\nWrite a factual concise legal summary (facts, question, reasoning and holding):\n'
PREFIX=enc(PROMPT_PREFIX); SUFFIX=enc(PROMPT_SUFFIX)
DOC_BUDGET=MAX_INPUT_TOKENS-len(PREFIX)-len(SUFFIX)-2
assert DOC_BUDGET>64

def chunk_prompts(document):
    return [[BOS]+PREFIX+c+SUFFIX for c in segments(document,DOC_BUDGET)]

def make_examples(records):
    examples=[]
    for rec in records:
        prompts=chunk_prompts(rec['document'])
        target=enc(rec['summary'])[:MAX_TARGET_TOKENS-1]+[EOS]
        # Chunk-specific training targets: summary supervision is document-level, not chunk-level.
        # Reusing full summary per chunk is a heuristic; compare against selection and report this limitation.
        for prompt in prompts:
            x=(prompt+target)[:-1]; y=([-100]*(len(prompt)-1)+target)
            assert len(x)==len(y) and len(x)<=cfg.rope_cache_len
            examples.append((x,y))
    return examples

def collate(batch):
    mx=max(len(x) for x,y in batch)
    xs=torch.full((len(batch),mx), PAD, dtype=torch.long)
    ys=torch.full((len(batch),mx),-100,dtype=torch.long)
    for i,(x,y) in enumerate(batch):xs[i,:len(x)]=torch.tensor(x);ys[i,:len(y)]=torch.tensor(y)
    return xs.to(DEVICE),ys.to(DEVICE)

rng=random.Random(SEED);indices=list(range(len(train_raw)));rng.shuffle(indices)
nval=max(1,round(len(indices)*VAL_FRAC));val_ids=set(indices[:nval]);train_docs=[train_raw[i] for i in indices if i not in val_ids];val_docs=[train_raw[i] for i in indices if i in val_ids]
assert train_docs, 'Need at least two training examples for holdout'
if MODE=='smoke':train_docs=train_docs[:SMOKE_TRAIN_DOCS];val_docs=val_docs[:SMOKE_VAL_DOCS]
train_ex=make_examples(train_docs);val_ex=make_examples(val_docs)
print('train docs/chunks',len(train_docs),len(train_ex),'val docs/chunks',len(val_docs),len(val_ex),'test',len(test_raw))

In [ ]:
import statistics as stt
def q(a,p): a=sorted(a); return a[min(len(a)-1,int(p*len(a)))]
def desc(nm,a): print(f"{nm:26s} n={len(a):>6} mean={stt.mean(a):>9.0f} med={stt.median(a):>9.0f} p90={q(a,.9):>9.0f} max={max(a):>9.0f}")
desc('train doc chars',[len(r['document']) for r in train_raw])
desc('test doc chars',[len(r['document']) for r in test_raw])
desc('train summary chars',[len(r['summary']) for r in train_raw])
S=random.Random(0).sample(train_raw,min(150,len(train_raw)))
t0=time.time()
dtok=[len(enc(r['document'])) for r in S]; stok=[len(enc(r['summary'])) for r in S]
nch=[len(chunk_prompts(r['document'])) for r in S]
print(f"tokenised {len(S)} docs in {time.time()-t0:.1f}s")
desc('doc tokens (sample)',dtok); desc('summary tokens (sample)',stok); desc('chunks/doc (sample)',nch)
print('summaries truncated at MAX_TARGET_TOKENS:',np.mean([l>MAX_TARGET_TOKENS-1 for l in stok]))
print('inference chunk coverage (max 4 chunks):',np.mean([min(4,c)/c for c in nch]))
print('train docs',len(train_docs),'-> train chunks',len(train_ex),'| val docs',len(val_docs),'val chunks',len(val_ex),'| test docs',len(test_raw))

## 7. From-scratch PEFT definitions — FIRST LINE: `class LowRankLinear`

In [ ]:
class LowRankLinear(nn.Module):
    # Frozen base W; additive LoRA delta = B A * alpha/r.
    def __init__(self,linear,r=8,alpha=16,kind='lora'):
        super().__init__();self.base=linear;self.kind=kind;self.r=r;self.alpha=alpha
        self.A=nn.Parameter(torch.empty(r,linear.in_features));self.B=nn.Parameter(torch.zeros(linear.out_features,r))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
        if kind=='adalora':
            self.importance=torch.zeros(r);self.active_rank=r
    def forward(self,x):
        a=self.A if self.kind!='adalora' else self.A[:self.active_rank]
        b=self.B if self.kind!='adalora' else self.B[:,:self.active_rank]
        return self.base(x)+F.linear(F.linear(x,a),b)*(self.alpha/self.r)

class MultiplicativeLinear(nn.Module):
    # LoRMA identity-inflated left multiplicative transformation:
    # W'=(I+BA)W, y=W x + B(A(Wx)).
    # This is the identity-inflation variant; not the paper's permutation-inflation variant.
    def __init__(self,linear,r=8,alpha=16):
        super().__init__();self.base=linear;d=linear.out_features;self.alpha=alpha;self.r=r
        self.A=nn.Parameter(torch.empty(r,d));self.B=nn.Parameter(torch.zeros(d,r))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
    def forward(self,x):
        y=self.base(x)
        return y + F.linear(F.linear(y,self.A),self.B)*(self.alpha/self.r)

class BottleneckAdapter(nn.Module):
    def __init__(self,d,bottleneck):
        super().__init__();self.down=nn.Linear(d,bottleneck);self.up=nn.Linear(bottleneck,d)
        nn.init.zeros_(self.up.weight);nn.init.zeros_(self.up.bias)
    def forward(self,x):return x+self.up(F.gelu(self.down(x)))

class AdapterBlock(nn.Module):
    # Original Block with additional residual adapter(s), no external adapter library.
    def __init__(self,base,method,cfg,bottleneck):
        super().__init__();self.base=base;self.method=method
        if method=='houlsby':self.post_attn=BottleneckAdapter(cfg.d_model,bottleneck)
        self.post_ffn=BottleneckAdapter(cfg.d_model,bottleneck)
    def forward(self,x,cos,sin,alibi_slopes,key_valid,past,use_cache):
        b=self.base
        h,new_cache=b.attn(b.attn_norm(x),cos,sin,alibi_slopes,key_valid,past,use_cache)
        x=x+h
        if self.method=='houlsby':x=self.post_attn(x)
        x=x+b.mlp(b.mlp_norm(x))
        x=self.post_ffn(x)
        return x,new_cache

def inject_method(model,method):
    for p in model.parameters():p.requires_grad=False
    if method in ('houlsby','pfeiffer'):
        for i,layer in enumerate(model.layers):model.layers[i]=AdapterBlock(layer,method,model.cfg,ADAPTER_BOTTLENECK)
    else:
        for layer in model.layers:
            # Map equivalent modules from the original code, without touching embeddings.
            for module in (layer.attn,layer.mlp):
                targets=('wq','wv') if module is layer.attn else ('w_down',)
                for name in targets:
                    base=getattr(module,name)
                    wrapped=MultiplicativeLinear(base,RANK) if method=='lorma' else LowRankLinear(base,RANK,kind=method)
                    setattr(module,name,wrapped)
    return model

def trainable_params(model):return [p for p in model.parameters() if p.requires_grad]

def update_adalora(model,global_step,total_steps):
    # Gradually prune low-importance rank components (simplified dynamic budget allocation).
    # Importance = EMA of |parameter * gradient|; no PEFT/AdaLoRA library.
    target=max(2,round(RANK*(1-.75*global_step/max(1,total_steps))))
    for m in model.modules():
        if isinstance(m,LowRankLinear) and m.kind=='adalora' and m.active_rank>target:
            if m.A.grad is None or m.B.grad is None:continue
            scores=(m.A.detach()*m.A.grad.detach()).abs().sum(1)+(m.B.detach()*m.B.grad.detach()).abs().sum(0)
            m.importance=.9*m.importance+.1*scores.cpu()
            # Reorder parameter columns/rows and optimizer moments not done; prune trailing components.
            # This is simplified rank-scheduled AdaLoRA-style, NOT full SVD reparameterization.
            m.active_rank=target

## 8. Training with best-checkpoint persistence — FIRST LINE: `def new_model`

In [ ]:
def new_model(method):
    m=LlamaLM(cfg)
    missing,unexpected=m.load_state_dict(initial_state,strict=True)
    assert not missing and not unexpected
    return inject_method(m,method).to(DEVICE)

def evaluate_loss(model, examples, limit=None):
    model.eval(); losses=[]
    with torch.no_grad():
        for example in examples[:limit]:
            x,y=collate([example])
            with torch.autocast('cuda',dtype=torch.float16,enabled=USE_AMP and DEVICE.type=='cuda'): loss=model(x,targets=y)
            losses.append(float(loss.item()))
    return sum(losses)/max(1,len(losses))

def train_one(method):
    print('\nMETHOD',method,flush=True);model=new_model(method)
    emb_w=model.tok_emb.weight                                   # NEW: tied with lm_head
    if TRAIN_SPECIAL_ROWS:                                       # NEW
        emb_w.requires_grad=True
        row_mask=torch.zeros(emb_w.shape[0],1,device=emb_w.device); row_mask[TRAIN_SPECIAL_ROWS]=1
        emb_w.register_hook(lambda g: g*row_mask.to(g.dtype))    # only the chosen rows get gradient
    adapter_params=[p for p in model.parameters() if p.requires_grad and p is not emb_w]  # NEW
    tr=sum(p.numel() for p in adapter_params)+len(TRAIN_SPECIAL_ROWS)*cfg.d_model         # NEW
    tot=sum(p.numel() for p in model.parameters()); print('trainable',tr,'total',tot,'percent',round(100*tr/tot,4))
    groups=[{'params':adapter_params,'weight_decay':WEIGHT_DECAY}]                        # NEW
    if TRAIN_SPECIAL_ROWS: groups.append({'params':[emb_w],'weight_decay':0.0})           # NEW
    opt=torch.optim.AdamW(groups,lr=LR)
    use_amp=USE_AMP and DEVICE.type=='cuda';scaler=torch.amp.GradScaler('cuda',enabled=use_amp)
    best=float('inf');log=[];start=time.time();step=0;win=[];stop=False
    n_ex=min(len(train_ex),MAX_EX_PER_EPOCH or len(train_ex))                             # NEW
    val_sub=random.Random(0).sample(val_ex,min(EVAL_LIMIT,len(val_ex)))                   # NEW
    total_steps=math.ceil(n_ex/BATCH_SIZE/GRAD_ACCUM)*EPOCHS
    for epoch in range(EPOCHS):
        rng.shuffle(train_ex);data=train_ex[:n_ex];model.train();opt.zero_grad(set_to_none=True);running=0.;nb=0
        for at in tqdm(range(0,n_ex,BATCH_SIZE),desc=f'{method} ep{epoch+1}'):
            x,y=collate(data[at:at+BATCH_SIZE])
            with torch.autocast('cuda',dtype=torch.float16,enabled=use_amp):loss=model(x,targets=y)/GRAD_ACCUM
            scaler.scale(loss).backward();lv=float(loss.item())*GRAD_ACCUM;running+=lv;win.append(lv);nb+=1
            if (at//BATCH_SIZE+1)%GRAD_ACCUM==0 or at+BATCH_SIZE>=n_ex:
                if method=='adalora':
                    scaler.unscale_(opt);update_adalora(model,step,total_steps)
                scaler.unscale_(opt) if method!='adalora' else None
                torch.nn.utils.clip_grad_norm_(trainable_params(model),1.0)
                scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True);step+=1
                if step%LOG_EVERY==0 or step==total_steps:                                # NEW block
                    el=time.time()-start;eta=el/step*(total_steps-step)
                    msg=f'[{method}] step {step}/{total_steps} train_loss {np.mean(win):.4f}';win.clear()
                    if EVAL_EVERY and step%EVAL_EVERY==0:
                        msg+=f' | val_loss({len(val_sub)} chunks) {evaluate_loss(model,val_sub):.4f}';model.train()
                    print(msg+f' | elapsed {el/60:.1f}m | ETA this method {eta/60:.1f}m',flush=True)
                if TIME_LIMIT_MIN and (time.time()-start)/60>TIME_LIMIT_MIN: stop=True;break  # NEW
        v=evaluate_loss(model,val_ex,limit=2 if MODE=='smoke' else VAL_LOSS_LIMIT)
        entry=dict(method=method,epoch=epoch+1,train_loss=running/max(1,nb),val_loss=v,seconds=time.time()-start)
        log.append(entry);print(entry)
        if v<best:
            best=v
            torch.save({'cfg':asdict(cfg),'method':method,'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},'metrics':entry,
                       'ranks':[m.active_rank for m in model.modules() if isinstance(m,LowRankLinear) and m.kind=='adalora']},os.path.join(OUT_DIR,f'{method}_best.pt'))
        if stop: print('time limit reached, stopping',method);break                       # NEW
    with open(os.path.join(OUT_DIR,method+'_history.json'),'w') as f:json.dump(log,f,indent=2)
    del model,opt;gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
    return dict(method=method,trainable=tr,total=tot,best_val_loss=best,seconds=time.time()-start)


## 9. Restore saved model and generate document-level summaries — FIRST LINE: `def load_best`

In [ ]:
def load_best(method):
    ck=torch.load(os.path.join(OUT_DIR,f'{method}_best.pt'),map_location='cpu',weights_only=False)
    m=new_model(method)
    m.load_state_dict(ck['model'],strict=True);m.eval()
    if ck.get('ranks'):
        for mod,r in zip([x for x in m.modules() if isinstance(x,LowRankLinear) and x.kind=='adalora'],ck['ranks']): mod.active_rank=r
    return m

@torch.no_grad()
def greedy(model,prompt,n_new):
    # No library generation calls. Prefix crop preserves last tokens; cache decoding.
    prompt=prompt[-(cfg.rope_cache_len-n_new-1):]
    x=torch.tensor([prompt],dtype=torch.long,device=DEVICE)
    with torch.autocast('cuda',enabled=USE_AMP and DEVICE.type=='cuda'):
        logits,cache=model(x,use_cache=True,start_pos=0)
        out=[]
        for j in range(n_new):
            nxt=int(logits[0,-1].argmax().item())
            if nxt==EOS:break
            out.append(nxt)
            logits,cache=model(torch.tensor([[nxt]],device=DEVICE),use_cache=True,cache=cache,start_pos=len(prompt)+j)
    return dec(out).strip()

def summarise(model,document):
    prompts=chunk_prompts(document)
    # Select capped, spread-across-document chunks in proportion to position; section coverage.
    k=min(4,len(prompts)) if MODE=='full' else min(2,len(prompts))
    chosen=np.unique(np.linspace(0,len(prompts)-1,k,dtype=int)).tolist()
    drafts=[greedy(model,prompts[i],SUMMARY_NEW_TOKENS//2) for i in chosen]
    if len(drafts)==1:return drafts[0]
    # Hierarchical rewrite of generated chunk drafts into one summary.
    second='\n'.join(drafts)
    final_prompt=chunk_prompts(second)[0]
    return greedy(model,final_prompt,SUMMARY_NEW_TOKENS)

def safe_summary(model,document):
    s=summarise(model,document).strip()
    if not s:  # evaluator rejects empty summaries; lead-words fallback
        s=' '.join(document.split()[:100])
    return s

def gen_batch(model, prompts, n_new, bs=GEN_BS):
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    outs = [None]*len(prompts)
    for s in range(0, len(order), bs):
        idx = order[s:s+bs]
        res = generate(model, [prompts[i] for i in idx], n_new, EOS, PAD,
                       temperature=0, rep_penalty=1.1, ban_eos=False)
        for i, r in zip(idx, res): outs[i] = dec(r).strip()
    return outs

def summarise_many(model, docs):
    summaries = [None]*len(docs); single_idx, single_p, flat, owner = [], [], [], []
    for di, d in enumerate(docs):
        ps = chunk_prompts(d)
        chosen = np.unique(np.linspace(0, len(ps)-1, min(K_CHUNKS, len(ps)), dtype=int)).tolist()
        if len(chosen) == 1: single_idx.append(di); single_p.append(ps[chosen[0]])
        else:
            for c in chosen: flat.append(ps[c]); owner.append(di)
    for di, s in zip(single_idx, gen_batch(model, single_p, FINAL_TOK)): summaries[di] = s
    joined = {}
    for di, t in zip(owner, gen_batch(model, flat, DRAFT_TOK)): joined.setdefault(di, []).append(t)
    mi = list(joined)
    fp = [chunk_prompts('\n'.join(joined[di]))[0] for di in mi]
    for di, s in zip(mi, gen_batch(model, fp, FINAL_TOK)): summaries[di] = s
    return [s if s.strip() else ' '.join(docs[i].split()[:100]) for i, s in enumerate(summaries)]

In [ ]:
'''def sync():
    if DEVICE.type=='cuda': torch.cuda.synchronize()
def bench(method,n=16):
    m=new_model(method); m.train(); use_amp=USE_AMP and DEVICE.type=='cuda'
    exs=random.Random(1).sample(train_ex,min(n+2,len(train_ex)))
    for i,ex in enumerate(exs):
        if i==2: sync(); t0=time.time()
        x,y=collate([ex])
        with torch.autocast('cuda',dtype=torch.float16,enabled=use_amp): loss=m(x,targets=y)
        loss.backward(); m.zero_grad(set_to_none=True)
    sync(); ts=(time.time()-t0)/(len(exs)-2)
    vx=random.Random(2).sample(val_ex,min(16,len(val_ex)))
    t1=time.time(); evaluate_loss(m,vx); sync(); tv=(time.time()-t1)/len(vx)
    del m; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return ts,tv
res={mt:bench(mt) for mt in METHODS}; print({k:(round(a,3),round(b,3)) for k,(a,b) in res.items()})
m=new_model('lora').eval(); docs=test_raw[:2]
sync(); t0=time.time(); [summarise(m,r['document']) for r in docs]; sync()
t_gen=(time.time()-t0)/len(docs); del m; gc.collect()   # untrained => no EOS => worst case
BUDGET_MIN=165
ts=np.mean([a for a,b in res.values()]); tv=np.mean([b for a,b in res.values()])
gen_s=t_gen*(len(test_raw)+len(val_docs))*len(METHODS)
avail=(BUDGET_MIN*60-gen_s-600)/len(METHODS)
print(f"sec/summary {t_gen:.1f} | generation total ~{gen_s/60:.0f} min | train+val budget per method ~{avail/60:.0f} min")
if avail>0:
    ex_ep=(avail/EPOCHS-len(val_ex)*tv)/ts
    print(f"full config needs {len(train_ex)*EPOCHS*ts/60:.0f} min/method; fits ~{int(ex_ep)} examples/epoch -> MAX_EX_PER_EPOCH={int(ex_ep)}, TIME_LIMIT_MIN={avail/60*0.9:.0f}")
else: print("Generation alone exceeds budget: cut chunks (k), SUMMARY_NEW_TOKENS, or batch generation (see end).")'''

In [ ]:

results=[]
for meth in METHODS:
    results.append(train_one(meth))
    # hf_push(os.path.join(OUT_DIR, f"{meth}_best.pt"),      f"{meth}/{meth}_best.pt")
    hf_push(os.path.join(OUT_DIR, f"{meth}_history.json"), f"{meth}/{meth}_history.json")
print(results)

In [ ]:
scorer=rouge_scorer.RougeScorer(['rouge1','rougeL'],use_stemmer=True)
try:
    from bert_score import BERTScorer
    bscorer=BERTScorer(lang='en',batch_size=16,device=str(DEVICE))
except Exception as e:
    bscorer=None; print('BERTScore unavailable:',e)
vdocs=val_docs[:VAL_GEN_DOCS]
val_truth={str(r['id']):r['summary'].strip() for r in vdocs}
json.dump(val_truth,open(os.path.join(OUT_DIR,'val_truth.json'),'w'),ensure_ascii=False)
metric_rows=[]
for method in METHODS:
    model=load_best(method)
    plist=summarise_many(model,[r['document'] for r in vdocs])
    del model;gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
    refs=[r['summary'].strip() for r in vdocs]
    json.dump({str(r['id']):p for r,p in zip(vdocs,plist)},open(os.path.join(OUT_DIR,f'val_pred_{method}.json'),'w'),ensure_ascii=False)
    sc=[scorer.score(a,b) for a,b in zip(refs,plist)]
    row={'method':method,'val_rouge1_f1':float(np.mean([s['rouge1'].fmeasure for s in sc])),
         'val_rougeL_f1':float(np.mean([s['rougeL'].fmeasure for s in sc])),
         'avg_pred_words':float(np.mean([len(p.split()) for p in plist]))}
    if bscorer is not None:
        _,_,f=bscorer.score(plist,refs); row['val_bertscore_f1']=float(f.mean())
    metric_rows.append(row); print(row)
print('avg reference words:',np.mean([len(v.split()) for v in val_truth.values()]))
if bscorer is not None: print('BEST by BERTScore F1:',max(metric_rows,key=lambda r:r['val_bertscore_f1'])['method'])
with open(os.path.join(OUT_DIR,'ablation_metrics.json'),'w') as f:json.dump({'training':results,'validation':metric_rows},f,indent=2)
hf_push(os.path.join(OUT_DIR,'ablation_metrics.json'),'ablation_metrics.json')

## 10. Generate all per-method submissions (FULL) or limited smoke checks — FIRST LINE: `for method in METHODS:`

In [ ]:
for method in METHODS:
    model=load_best(method)
    docs=test_raw if MODE=='full' else test_raw[:SMOKE_TEST_DOCS]
    sums=summarise_many(model,[r['document'] for r in docs])
    predictions=[{'id':str(r['id']),'summary':s} for r,s in zip(docs,sums)]
    assert all(isinstance(p['id'],str) and p['summary'].strip() for p in predictions), 'bad predictions'
    if MODE=='full':
        assert len(predictions)==len(test_raw) and len({str(x['id']) for x in predictions})==len(test_raw)
    path=os.path.join(OUT_DIR,f'phase6_{method}_{MODE}.json')
    with open(path,'w',encoding='utf8') as f:json.dump(predictions,f,ensure_ascii=False,indent=2)
    print('Wrote',path,len(predictions))
    hf_push(path, f"{method}/{os.path.basename(path)}")
    del model;gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
print('Completed',MODE,'mode. For official submission use FULL outputs only.')

## 11. Interpretation and competition compliance

- Houlsby: serial bottleneck adapters after attention and feed-forward.
- Pfeiffer: one bottleneck adapter after feed-forward.
- LoRA: additive low-rank frozen-linear updates.
- AdaLoRA: simplified gradient-importance-based rank scheduling (not paper-faithful SVD reallocation; this needs further implementation for a faithful AdaLoRA comparison).
- LoRMA: identity-inflated multiplicative update `(I + BA)W` (not the permutation variant).
- Chunk-level supervision uses whole document reference for every chunk; this is weak supervision and can hallucinate absent facts. Report this explicitly.
- **Absent:** official BERTScore (do not use external pretrained weights), knowledge distillation, experiments on all four Phase-1 architectures. Those must be added to meet the complete official Phase-6 brief.
- Notebook outputs are unexecuted until you provide your data, tokenizer, authentic Phase-1 checkpoint and GPU environment.